# Paper pipeline — native notebook, live progress

Runs every experiment **in-process** (no subprocess wrappers): progress bars
per model / per example, live summaries after each stage, incremental CSVs so
any cell can be interrupted and rerun (finished work is skipped).

Stages: cohorts → obstruction → repairs → sensitivity → bottleneck →
extraction/two-hop → toy ontogeny → aggregate. Read results in
`analyze_results.ipynb`.

In [ ]:
# ---------------- config ----------------
MODELS = [
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B",
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B",
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-7B",
    "allenai/OLMo-7B-hf",
]
ONLY     = ""        # substring filter ("3.2", "OLMo", ...); "" = all
DEVICE   = "cuda"
STANDIN  = True      # False once cohorts carry pipeline verdict labels
C_PLUS_Q = 0.10
N_TERTILE = 170      # PopQA records per popularity tertile
BALANCE  = 60        # failures/successes per cohort

import os, gc, json, torch, pandas as pd
from tqdm.auto import tqdm
from collections import Counter
torch.set_grad_enabled(False)
pd.set_option("display.precision", 3)

from frozen_cache import (Weights, build_cache, certify_frozen,
                          target_delivery, tdla_edge_scores)
from obstruction import solve_obstruction
from run_obstruction_validation import auc, ablation_effect
from repairs import (repair_presence, repair_transport,
                     repair_transport_force, repair_selection,
                     repair_random_heads, certify_wrappers, margin)
from build_cohort import (process, load_popqa, make_extraction, make_twohop,
                          find_span)

models = [m for m in MODELS if ONLY.lower() in m.lower()]
tag = lambda m: m.split("/")[-1].replace(".", "").lower()
for d in ("cohorts", "results"):
    os.makedirs(d, exist_ok=True)
print(f"{len(models)} models selected:", [tag(m) for m in models])

_loaded = {}
def get_model(name):
    """One model resident at a time; drops stale notebook globals (they pin
    the previous model) and streams shards to avoid 2x host-RAM peaks."""
    if name in _loaded:
        return _loaded[name]
    import __main__
    for v in ("model", "tok", "W", "C", "D", "df"):
        if hasattr(__main__, v):
            delattr(__main__, v)
    _loaded.clear()
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, torch_dtype=torch.float32, low_cpu_mem_usage=True,
        attn_implementation="eager").to(DEVICE).eval()
    _loaded[name] = (model, tok, Weights(model))
    return _loaded[name]

def load_records(m):
    """Cohort records with verdicts guaranteed: backfills from the model's
    obstruction.csv (by idx) when the cohort file predates verdict assignment
    (resume path), then falls back to correct/unlabeled."""
    records = [json.loads(l) for l in open(cohorts[m])]
    if any(r.get("verdict") is None for r in records):
        obcsv = f"results/{tag(m)}/obstruction.csv"
        if os.path.exists(obcsv):
            vd = dict(pd.read_csv(obcsv)[["idx", "verdict"]].values)
            for i, r in enumerate(records):
                r["verdict"] = r.get("verdict") or vd.get(i)
    for r in records:
        if r.get("verdict") is None:
            r["verdict"] = "correct" if r["correct"] else "unlabeled"
    return records

## 0. Certificate check (30 s)
One prompt per model: C1/C2 must pass before anything else is worth running.

In [ ]:
for m in tqdm(models, desc="certificates"):
    model, tok, W = get_model(m)
    ids = tok("The capital of France is",
              return_tensors="pt").input_ids[0].to(DEVICE)
    C = build_cache(model, W, ids)
    err = certify_frozen(W, C)
    print(f"  {tag(m)}: C1/C2 pass (rel err {err:.1e}, "
          f"L={W.L}, center={W.center})")

## 1. Cohorts (parametric, PopQA)
Greedy generation per record — the slow stage. Incremental: appends to the
jsonl as it goes and resumes mid-cohort.

In [ ]:
def build_parametric(m):
    path = f"cohorts/{tag(m)}_parametric.jsonl"
    done = sum(1 for _ in open(path)) if os.path.exists(path) else 0
    src = load_popqa(N_TERTILE)
    if done >= min(len(src), 2 * BALANCE):
        return path
    model, tok, W = get_model(m)
    raw_path = path + ".raw"
    seen = sum(1 for _ in open(raw_path)) if os.path.exists(raw_path) else 0
    with open(raw_path, "a") as f:
        bar = tqdm(src[seen:], desc=f"{tag(m)} generate", initial=seen,
                   total=len(src))
        nf = sum(1 for l in open(raw_path)
                 if not json.loads(l)["correct"]) if seen else 0
        for r in bar:
            try:
                rec = process(model, tok, DEVICE, r["prompt"], r["subject"],
                              r["answer"], r.get("aliases", []))
            except (ValueError, IndexError):
                continue
            f.write(json.dumps(rec) + "\n"); f.flush()
            nf += not rec["correct"]
            bar.set_postfix(failures=nf)
    out = [json.loads(l) for l in open(raw_path)]
    clean = [o for o in out if not o["copy_contaminated"]]
    import random as _r; rng = _r.Random(0)
    fails = [o for o in clean if not o["correct"]]
    succs = [o for o in clean if o["correct"]]
    cohort = (rng.sample(fails, min(BALANCE, len(fails))) +
              rng.sample(succs, min(BALANCE, len(succs))))
    with open(path, "w") as f:
        for o in cohort:
            f.write(json.dumps(o) + "\n")
    print(f"  {tag(m)}: {len(fails)} failures / {len(succs)} successes "
          f"-> balanced {len(cohort)}")
    return path

cohorts = {}
for m in models:
    cohorts[m] = build_parametric(m)

## 2. Obstruction ob(s) + stand-in verdicts
Per model: caches (bar 1), c⁺ calibration, verdicts if needed, solver (bar 2
with live ob readout). Writes `results/<tag>/obstruction.csv`; skips if done.

In [ ]:
def run_obstruction(m):
    out = f"results/{tag(m)}"; os.makedirs(out, exist_ok=True)
    csv_path = f"{out}/obstruction.csv"
    if os.path.exists(csv_path):
        return pd.read_csv(csv_path)
    model, tok, W = get_model(m)
    records = [json.loads(l) for l in open(cohorts[m])]
    caches, delivered, pis = {}, [], []
    ug = lambda C, t: ((C.inv_f[-1] * W.ln_f) * W.WU[t]).squeeze()
    def pi_S(C, t, S):
        lo, hi = W.L // 4, 3 * W.L // 4
        return max(float((C.resid[l][j] * ug(C, t)).sum())
                   for l in range(lo, hi + 1) for j in S)
    for i, r in enumerate(tqdm(records, desc=f"{tag(m)} caches")):
        ids = tok(r["prompt"], return_tensors="pt").input_ids[0].to(DEVICE)
        C = build_cache(model, W, ids); certify_frozen(W, C)
        caches[i] = (ids, C)
        if r["correct"]:
            delivered.append(float(target_delivery(
                C.resid[-1], W, C, r["target_first_token"])))
    c_plus = float(torch.tensor(delivered).quantile(C_PLUS_Q))
    if STANDIN:
        th_d = c_plus
        th_p = torch.tensor([pi_S(caches[i][1], r["target_first_token"],
                             range(r["source_span"][0], r["source_span"][1]+1))
                             for i, r in enumerate(records) if r["correct"]]
                            ).quantile(C_PLUS_Q).item()
        for i, r in enumerate(records):
            if r["correct"]: r["verdict"] = "correct"; continue
            S = range(r["source_span"][0], r["source_span"][1] + 1)
            d = float(target_delivery(caches[i][1].resid[-1], W, caches[i][1],
                                      r["target_first_token"]))
            p = pi_S(caches[i][1], r["target_first_token"], S)
            r["verdict"] = ("selection" if d >= th_d else
                            "transport" if p >= th_p else "presence")
    rows = []
    bar = tqdm(records, desc=f"{tag(m)} ob(s)")
    for i, r in enumerate(bar):
        ids, C = caches[i]
        S = list(range(r["source_span"][0], r["source_span"][1] + 1))
        res = solve_obstruction(W, C, S, r["target_first_token"], c_plus)
        eff = (ablation_effect(model, ids, W, C, r["target_first_token"], S)
               if r["verdict"] != "correct" else None)
        rows.append(dict(idx=i, verdict=r["verdict"], ob=res.ob,
                         ob_norm=res.ob_norm, centroid=res.depth_centroid,
                         gap0=res.delivered_gap0, pi_S=res.pi_S,
                         ablation_effect=eff, cg_iters=res.cg_iters))
        bar.set_postfix(verdict=r["verdict"][:4], ob=f"{res.ob:.2f}")
    df = pd.DataFrame(rows); df.to_csv(csv_path, index=False)
    with open(cohorts[m], "w") as f:      # persist verdicts back
        for r in records: f.write(json.dumps(r) + "\n")
    return df

for m in models:
    df = run_obstruction(m)
    print(f"== {tag(m)} ==")
    display(df.groupby("verdict").ob.agg(["median", "count"]).round(3))

## 3. Repair matrix (causal validation)

In [ ]:
def run_repairs(m):
    out = f"results/{tag(m)}/repair_matrix.csv"
    if os.path.exists(out):
        return pd.read_csv(out)
    model, tok, W = get_model(m)
    records = load_records(m)
    fails = [r for r in records if r["verdict"] != "correct"
             and r["competitor_token"] != -1]
    certify_wrappers(model, W, tok(fails[0]["prompt"],
                     return_tensors="pt").input_ids[0].to(DEVICE))
    BAND = list(range(int(0.2 * W.L), int(0.6 * W.L)))
    rows = []
    bar = tqdm(fails, desc=f"{tag(m)} repairs")
    for r in bar:
        ids = tok(r["prompt"], return_tensors="pt").input_ids[0].to(DEVICE)
        C = build_cache(model, W, ids)
        g, c = r["target_first_token"], r["competitor_token"]
        S = list(range(r["source_span"][0], r["source_span"][1] + 1))
        m0, _ = margin(model, ids, g, c)
        row = dict(verdict=r["verdict"], margin_base=m0)
        if r.get("donor_prompt"):
            d_ids = tok(r["donor_prompt"],
                        return_tensors="pt").input_ids[0].to(DEVICE)
            D = build_cache(model, W, d_ids)
            dS = list(range(r["donor_source_span"][0],
                            r["donor_source_span"][1] + 1))
            if len(dS) == len(S):
                donor = {l: D.resid[l][dS] for l in BAND}
                mm, fl = repair_presence(model, ids, g, c, S, donor, BAND)
                row["presence_patch"], row["presence_patch_flip"] = mm - m0, fl
        mm, fl = repair_transport(model, W, C, ids, g, c, S, k=8)
        row["transport_edges"], row["transport_edges_flip"] = mm - m0, fl
        mm, fl = repair_transport_force(model, W, C, ids, g, c, S,
                                        k=8, alpha=1.0)
        row["transport_force"], row["transport_force_flip"] = mm - m0, fl
        mm, fl = repair_selection(model, W, C, ids, g, c, k=4)
        row["selection_demoters"], row["selection_demoters_flip"] = mm - m0, fl
        mm, fl = repair_random_heads(model, W, ids, g, c, k=4, seed=len(rows))
        row["random_heads"], row["random_heads_flip"] = mm - m0, fl
        rows.append(row)
        bar.set_postfix(v=str(r["verdict"])[:4],
                sel=f"{row['selection_demoters']:+.1f}")
    df = pd.DataFrame(rows); df.to_csv(out, index=False)
    return df

for m in models:
    df = run_repairs(m)
    print(f"== {tag(m)} repair matrix (mean margin change) ==")
    cols = [c for c in ("presence_patch", "transport_edges",
                        "transport_force", "selection_demoters",
                        "random_heads") if c in df]
    display(df.groupby("verdict")[cols].mean().round(2))

## 4. Sensitivity grid

In [ ]:
from run_sensitivity import sweep
for m in models:
    out = f"results/{tag(m)}/sensitivity.csv"
    if os.path.exists(out):
        continue
    model, tok, W = get_model(m)
    records = load_records(m)
    res = pd.DataFrame(sweep(model, tok, W, records, DEVICE))
    res.to_csv(out, index=False)
    print(f"{tag(m)}: min agreement {res.agree_with_ref.min():.2f}, "
          f"AUC [{res.auc_pt_vs_sc.min():.3f}, {res.auc_pt_vs_sc.max():.3f}]")

## 5. Bottleneck (word-level forced decoding)

In [ ]:
from run_bottleneck import forced_trajectory
for m in models:
    out = f"results/{tag(m)}/bottleneck.csv"
    if os.path.exists(out):
        continue
    model, tok, W = get_model(m)
    records = [r for r in load_records(m) if r.get("target_text")]
    rows = []
    for r in tqdm(records, desc=f"{tag(m)} bottleneck"):
        t = forced_trajectory(model, tok, r["prompt"], r["target_text"], DEVICE)
        t["verdict"] = r["verdict"]; rows.append(t)
    df = pd.DataFrame(rows); df.to_csv(out, index=False)
    d = df[df.diverged]
    display(d.groupby("verdict").agg(rails=("back_on_rails", "mean"),
                                     n=("div_step", "size")).round(2))

## 6. Extraction & two-hop
Prediction: extraction has **zero presence failures**; failures split
transport/selection. Presence is fixed by token identity, so the stand-in
rule here only distinguishes delivered (selection) vs not (transport).

In [ ]:
for m in models:
    model, tok, W = get_model(m)
    for task, mk in (("extraction", lambda: make_extraction(n=120)),
                     ("twohop", make_twohop)):
        out_dir = f"results/{tag(m)}_{task}"; os.makedirs(out_dir, exist_ok=True)
        csv_path = f"{out_dir}/obstruction.csv"
        if os.path.exists(csv_path):
            continue
        rows = []
        c_del = []
        recs = []
        for r in tqdm(mk(), desc=f"{tag(m)} {task}"):
            try:
                rec = process(model, tok, DEVICE, r["prompt"], r["subject"],
                              r["answer"], r.get("aliases", []))
            except (ValueError, IndexError):
                continue
            ids = tok(rec["prompt"], return_tensors="pt").input_ids[0].to(DEVICE)
            C = build_cache(model, W, ids)
            d = float(target_delivery(C.resid[-1], W, C,
                                      rec["target_first_token"]))
            recs.append((rec, C, d))
            if rec["correct"]: c_del.append(d)
        if len(c_del) < 5:
            print(f"  {tag(m)} {task}: too few successes, skipped"); continue
        c_plus = float(torch.tensor(c_del).quantile(C_PLUS_Q))
        for rec, C, d in tqdm(recs, desc=f"{tag(m)} {task} ob"):
            S = list(range(rec["source_span"][0], rec["source_span"][1] + 1))
            v = ("correct" if rec["correct"] else
                 "selection" if d >= c_plus else "transport")
            res = solve_obstruction(W, C, S, rec["target_first_token"], c_plus)
            rows.append(dict(verdict=v, ob=res.ob, gap0=res.delivered_gap0,
                             centroid=res.depth_centroid, task=task))
        df = pd.DataFrame(rows); df.to_csv(csv_path, index=False)
        print(f"== {tag(m)} {task} ==",
              dict(Counter(df[df.verdict != 'correct'].verdict)))

## 7. Toy ontogeny (controlled exposure) — 3 seeds, live loss bar

In [ ]:
from train_toy import make_world, make_corpus, make_model, diagnose
for seed in (0, 1, 2):
    out = f"results/toy_seed{seed}"; os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/toy_ontogeny.csv"):
        continue
    vocab, tid, facts = make_world(seed=seed)
    sents = make_corpus(vocab, tid, facts, seed=seed)
    model = make_model(len(vocab), DEVICE)
    STEPS = 3000
    ckpts = sorted({int(STEPS * f) for f in (0.02, .05, .1, .2, .4, .7, 1.0)})
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    data = torch.zeros(len(sents), 6, dtype=torch.long)
    for i, s in enumerate(sents): data[i, :len(s)] = torch.tensor(s[:6])
    g = torch.Generator().manual_seed(seed)
    all_rows = []
    model.train()
    with torch.enable_grad():
        bar = tqdm(range(1, STEPS + 1), desc=f"toy seed {seed}")
        for step in bar:
            x = data[torch.randint(0, len(sents), (64,), generator=g)].to(DEVICE)
            loss = model(x, labels=x).loss
            loss.backward(); opt.step(); opt.zero_grad()
            bar.set_postfix(loss=f"{loss.item():.3f}")
            if step in ckpts:
                model.eval()
                rows = diagnose(model, tid, facts, DEVICE)
                for r in rows: r["step"] = step
                all_rows += rows
                acc = sum(r["correct"] for r in rows) / len(rows)
                fails = Counter(r["verdict"] for r in rows if not r["correct"])
                bar.write(f"  step {step}: acc {acc:.2f} {dict(fails)}")
                model.train()
    model.eval()
    pd.DataFrame(all_rows).to_csv(f"{out}/toy_ontogeny.csv", index=False)

### Toy predictions P1-P3

In [ ]:
import glob, matplotlib.pyplot as plt
frames = [pd.read_csv(p).assign(seed=i) for i, p in
          enumerate(sorted(glob.glob("results/toy_seed*/toy_ontogeny.csv")))]
if frames:
    toy = pd.concat(frames)
    f = toy[~toy.correct]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
    sh = f.groupby(["step", "verdict"]).size().unstack(fill_value=0)
    sh.div(sh.sum(1), axis=0).plot(ax=ax[0], marker="o")
    ax[0].set_xscale("log"); ax[0].set_title("P2: emergence order")
    f = f.assign(band=pd.qcut(f.exposure, 3, labels=["low", "mid", "high"],
                              duplicates="drop"))
    (f.groupby(["band", "verdict"]).size().unstack(fill_value=0)
       .pipe(lambda d: d.div(d.sum(1), axis=0))).plot(kind="bar", ax=ax[1])
    ax[1].set_title("P1: exposure -> class")
    (f.groupby(["crowded", "verdict"]).size().unstack(fill_value=0)
       .pipe(lambda d: d.div(d.sum(1), axis=0))).plot(kind="bar", ax=ax[2])
    ax[2].set_title("P3: crowding -> selection")
    plt.tight_layout(); plt.savefig("results/toy_predictions.png", dpi=150)
    plt.show()

## 8. Aggregate
Cross-model tables/figures to `results/aggregate/`; then open
`analyze_results.ipynb` for the readable report.

In [ ]:
import aggregate_results
aggregate_results.main()